# $D^0 \to \pi^+\pi^-\pi^+\pi^-$: the BESIII model

The amplitude model of BESIII, Chin. Phys. C 48 (2024) 083001
([arXiv:2312.02524](https://arxiv.org/abs/2312.02524)), from a joint analysis of
$D^0 \to \pi^+\pi^-\pi^+\pi^-$ and $\pi^+\pi^-\pi^0\pi^0$; `aman_examples.d024pi` builds
its $\pi^+\pi^-\pi^+\pi^-$ amplitudes in the paper's formalism (covariant Zemach tensors,
unnormalised Blatt-Weisskopf factors, Gounaris-Sakurai, the BaBar $\pi\pi$ K-matrix with
its two-S-wave expansion, $3\pi$ running widths of $a_1(1260)$, $a_1(1640)$ and
$\pi(1300)$). Where the paper leaves a convention open, it was fixed by reproducing the
published fit fractions, interference fractions and CP-even fraction $F_+$; see the
module documentation. The amplitudes agree with an independent NumPy implementation to
$10^{-13}$ (`tests/test_d024pi.py`).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from tensorwaves.data.rng import NumpyUniformRNG
from tensorwaves.function.sympy import create_function

from aman_examples import d024pi

## Model

One chain per $D^0$ coupling; the S-wave terms ($\pi\pi$ S-wave with $\rho^0$, $f_2$ or
another S-wave) have one chain per production-vector coefficient. All chains are summed
over the exchanges of the identical pions.

In [ ]:
model = d024pi.formulate_model()
for name in model.chains:
    print(f"{name:20s} {model.parameter_defaults[model.couplings[name]]:.4g}")

## Fit fractions and $F_+$

On uniform phase space, against the $\pi^+\pi^-\pi^+\pi^-$ column of the paper's
Table 5 (total uncertainties). $F_+$ uses $\bar{A}(p) = A(\bar{p})$, with $\bar{p}$ the
CP-conjugate event (charges exchanged, three-momenta reversed).

In [ ]:
phsp = d024pi.phase_space_generator().generate(100_000, NumpyUniformRNG(seed=1))
parity = {"x": -1, "y": -1, "z": -1, "E": 1}
cp = {
    f"p{i}_{c}": parity[c] * phsp[f"p{j}_{c}"]
    for i, j in enumerate((1, 0, 3, 2))
    for c in "xyzE"
}
amps, amps_bar = {}, {}
for name, expr in model.chains.items():
    function = create_function(expr.doit(), backend="jax")
    coupling = model.parameter_defaults[model.couplings[name]]
    amps[name] = coupling * np.asarray(function(phsp))
    amps_bar[name] = coupling * np.asarray(function(cp))
total, total_bar = sum(amps.values()), sum(amps_bar.values())
norm = np.mean(np.abs(total) ** 2)
print(f"{'':20s} {'this model':>10s}   paper")
for group, names in d024pi.fit_fraction_groups().items():
    fraction = 100 * np.mean(np.abs(sum(amps[n] for n in names)) ** 2) / norm
    value, error = d024pi.FIT_FRACTIONS[group]
    print(f"{group:20s} {fraction:9.1f}%   {value:5.1f} ± {error}%")
plus = np.mean(np.abs(total + total_bar) ** 2)
minus = np.mean(np.abs(total - total_bar) ** 2)
print(
    f"F+ = {100 * plus / (plus + minus):.1f}%   (paper: {d024pi.F_PLUS[0]} ± {d024pi.F_PLUS[1]}%)"
)

## Projections

In [ ]:
def mass(sample, indices):
    p = sum(np.column_stack([sample[f"p{i}_{c}"] for c in "xyzE"]) for i in indices)
    return np.sqrt(p[:, 3] ** 2 - np.sum(p[:, :3] ** 2, axis=1))


weights = np.abs(total) ** 2
projections = [
    (r"$m(\pi^+\pi^-)$", [(0, 1), (0, 3), (2, 1), (2, 3)]),
    (r"$m(\pi^+\pi^+\pi^-)$", [(0, 1, 2), (0, 2, 3)]),
    (r"$m(\pi^-\pi^-\pi^+)$", [(0, 1, 3), (1, 2, 3)]),
]
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), tight_layout=True)
for ax, (label, combinations) in zip(axes, projections, strict=True):
    values = np.concatenate([mass(phsp, c) for c in combinations])
    w = np.tile(weights, len(combinations))
    ax.hist(values, bins=80, weights=w / w.sum(), histtype="step", label=r"$|A|^2$")
    ax.hist(
        values,
        bins=80,
        weights=np.full(len(values), 1 / len(values)),
        histtype="step",
        label="phase space",
    )
    ax.set_xlabel(f"{label} [GeV]")
axes[0].legend()
plt.show()